In [ ]:
from pathlib import Path
from zipfile import ZipFile
import requests

class Client:
    def __init__(self, url):
        if url[-1] != '/':
            url = url + '/'
        self.base_url = url
        
    def get_json(self, get, params={}):
        params['format'] = 'json'
        url = self.base_url + get

        return requests.get(url, params=params).json()
        
    def get_image(self,
                  series_instance_uid,
                  local_path,
                  unzip,
                  remove_zip):
        
        url = self.base_url + 'getImage'
        local_path = Path(local_path)
        local_path.parent.mkdir(parents=True)
        
        params = {'SeriesInstanceUID': series_instance_uid}
        with requests.get(url, params=params, stream=True) as req:
            req.raise_for_status()
            with open(local_path, 'wb') as ff:
                for chunk in req.iter_content(chunk_size=8192): 
                    ff.write(chunk)
        if unzip:
            with ZipFile(local_path, 'r') as zz:
                try:
                    zz.extractall(local_path.parent)
                except:
                    print(f'unzipping for {local_path} failed')
            if remove_zip:
                local_path.unlink()
        return True


In [ ]:
from pathlib import Path
import pandas as pd
##################################### parameter #############################################
base_url = 'https://services.cancerimagingarchive.net/services/v4/TCIA/query'
collection = 'HEAD-NECK-RADIOMICS-HN1'
output_dir = 'D:\hvp'
##################################### parameter #############################################

client = Client(base_url)
output_dir = Path(output_dir)

series_df = pd.DataFrame(data=client.get_json('getSeries', {'Collection': collection}))
series_df = series_df[['StudyInstanceUID', 'SeriesInstanceUID', 'Modality']]

patient_df = pd.DataFrame(data=client.get_json('getPatientStudy', {'Collection': collection}))
patient_df = patient_df[['PatientID', 'StudyInstanceUID']]

# get the StudyUIDs of all CT's with a corresponding RTSTRUCT file
rts_stud_uids = series_df[series_df['Modality'] == 'RTSTRUCT']['StudyInstanceUID'].values
# use only CT cases with a corresponding RTSTRUCT file
series_df = series_df[
    (
        series_df['Modality'].isin(['CT', 'RTSTRUCT', 'SEG'])
    ) & (
        series_df['StudyInstanceUID'].isin(rts_stud_uids)
    )
]

download_df = patient_df.merge(series_df, how='right', on='StudyInstanceUID')

for ii, row in download_df.iterrows():
    uid = row['SeriesInstanceUID']
    
    target_dir = (output_dir / row['PatientID']) / row['Modality']
    
    # check if target_dir already exists, if so append a count number
    if target_dir.exists():
        split = target_dir.stem.split('_')
        if len(split) == 2:
            count = int(split[-1]) + 1
        else:
            count = 1
        target_dir = target_dir.parent / f'{target_dir.stem}_{count}'
    print(f'downloading: {target_dir}')

    target_path = target_dir / (row['PatientID'] + '.zip')
        
    client.get_image(uid, target_path, unzip=True, remove_zip=True)


In [ ]:
from pathlib import Path
import pandas as pd
from TCIAClient import Client

##################################### parameter #############################################
base_url = 'https://services.cancerimagingarchive.net/services/v4/TCIA/query'
collection = 'OPC-Radiomics' 
output_dir = 'D:/hvp/' + collection
##################################### parameter #############################################

client = Client(base_url)
output_dir = Path(output_dir)

series_df = pd.DataFrame(data=client.get_json('getSeries', {'Collection': collection}))
series_df = series_df[['StudyInstanceUID', 'SeriesInstanceUID', 'Modality']]

patient_df = pd.DataFrame(data=client.get_json('getPatientStudy', {'Collection': collection}))
patient_df = patient_df[['PatientID', 'StudyInstanceUID', 'SeriesCount']]

# get the StudyUIDs of all CT's with a corresponding RTSTRUCT file
rts_stud_uids = series_df[series_df['Modality'] == 'RTSTRUCT']['StudyInstanceUID'].values
# use only CT cases with a corresponding RTSTRUCT file
series_df = series_df[
    (
        series_df['Modality'].isin(['CT', 'RTSTRUCT'])
    ) & (
        series_df['StudyInstanceUID'].isin(rts_stud_uids)
    )
]

download_df = patient_df.merge(series_df, how='right', on='StudyInstanceUID')

for ii, row in download_df.iterrows():
    uid = row['SeriesInstanceUID']
    
    target_dir = (output_dir / row['PatientID']) / row['Modality']
    
    # check if target_dir already exists, if so append a count number
    if target_dir.exists():
        split = target_dir.stem.split('_')
        if len(split) == 2:
            count = int(split[-1]) + 1
        else:
            count = 1
        target_dir = target_dir.parent / f'{target_dir.stem}_{count}'
    print(f'downloading: {target_dir}')

    target_path = target_dir / (row['PatientID'] + '.zip')
        
    client.get_image(uid, target_path, unzip=True, remove_zip=True)
